In [2]:
import pickle
from pathlib import Path

import requests

COOKIES_FILE = Path("ceneo_cookies.pkl")

def load_or_create_session(product_id: str) -> requests.Session:
    session = requests.Session()

    # Pick one browser fingerprint for the whole session.
    # Rotating User-Agent mid-session often breaks pagination / triggers bot checks.
    session.headers.update(random_headers())

    if COOKIES_FILE.exists():
        with open(COOKIES_FILE, "rb") as f:
            session.cookies.update(pickle.load(f))
        print("Loaded existing cookies")
    else:
        session.get(f"{BASE_URL}/{product_id}", timeout=10)
        with open(COOKIES_FILE, "wb") as f:
            pickle.dump(session.cookies, f)
        print("Fresh session, cookies saved")

    return session

In [3]:
import base64
import time
import random
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.ceneo.pl"

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4.1 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
]

ACCEPT_LANGUAGES = [
    "pl-PL,pl;q=0.9,en-US;q=0.8,en;q=0.7",
    "pl-PL,pl;q=0.9,en;q=0.8",
    "pl,en-US;q=0.9,en;q=0.8",
    "pl-PL,pl;q=1.0",
]


def random_headers() -> dict:
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": random.choice(ACCEPT_LANGUAGES),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Referer": "https://www.ceneo.pl/",
    }


def get_total_pages(soup: BeautifulSoup) -> int:
    page_numbers = []
    for btn in soup.select(".pagination__item[data-hash]"):
        text = btn.get_text(strip=True)
        if text.isdigit():
            page_numbers.append(int(text))
    active = soup.select_one(".pagination__item.active span")
    if active and active.get_text(strip=True).isdigit():
        page_numbers.append(int(active.get_text(strip=True)))
    return max(page_numbers) if page_numbers else 1


def parse_reviews(soup: BeautifulSoup) -> list[dict]:
    reviews = []
    for card in soup.select(".js_product-review"):
        score_el = card.select_one(".user-post__score-count")
        text_el = card.select_one(".user-post__text")
        rec_el = card.select_one(".user-post__author-recomendation em")
        time_tags = card.select("time[datetime]")
        pros = [el.get_text(strip=True) for el in card.select(".review-feature__item--positive")]
        votes_yes = card.select_one(".js_vote-yes span")
        votes_no = card.select_one(".js_vote-no span")

        reviews.append({
            "score": score_el.get_text(strip=True) if score_el else None,
            "text": text_el.get_text(strip=True) if text_el else None,
            "recommended": rec_el.get_text(strip=True) if rec_el else None,
            "date": time_tags[0]["datetime"] if time_tags else None,
            "usage_period": time_tags[1].get_text(strip=True) if len(time_tags) > 1 else None,
            "pros": pros,
            "votes_yes": int(votes_yes.get_text(strip=True)) if votes_yes else 0,
            "votes_no": int(votes_no.get_text(strip=True)) if votes_no else 0,
        })
    return reviews


def fetch_soup(session: requests.Session, url: str) -> BeautifulSoup:
    response = session.get(url, timeout=20, allow_redirects=True)
    response.raise_for_status()

    html = response.text

    # Cloudflare Turnstile / bot-check page (you'll see form action "/Captcha/Add").
    if ("cf-turnstile" in html) or ("/Captcha/Add" in html) or ("challenges.cloudflare.com/turnstile" in html):
        raise RuntimeError(
            "Ceneo returned a Cloudflare Turnstile captcha page instead of reviews. "
            "This is why you get 0 reviews on later pages. "
            f"URL requested: {url} | final URL: {response.url}"
        )

    return BeautifulSoup(html, "html.parser")


def scrape_all_reviews(product_id: str) -> list[dict]:
    session = load_or_create_session(product_id)

    # Warm up — hit product page first to get cookies
    session.get(f"{BASE_URL}/{product_id}", timeout=10)
    time.sleep(random.uniform(1.0, 2.5))

    # Fetch page 1 to determine total pages
    first_url = f"{BASE_URL}/{product_id}/opinie-1"
    soup = fetch_soup(session, first_url)
    total_pages = get_total_pages(soup)
    print(f"Total pages: {total_pages}")

    all_reviews = []

    for page in range(1, total_pages + 1):
        if page > 1:
            url = f"{BASE_URL}/{product_id}/opinie-{page}"
            soup = fetch_soup(session, url)

        reviews = parse_reviews(soup)
        all_reviews.extend(reviews)
        print(f"Page {page}/{total_pages}: {len(reviews)} reviews")

        if page < total_pages:
            time.sleep(random.uniform(1.5, 4.0))

    print(f"\nDone. Total reviews scraped: {len(all_reviews)}")
    return all_reviews


reviews = scrape_all_reviews("148187174")

Loaded existing cookies
Total pages: 4
Page 1/4: 11 reviews
Page 2/4: 0 reviews
Page 3/4: 0 reviews
Page 4/4: 0 reviews

Done. Total reviews scraped: 11


In [10]:
reviews

[{'score': None,
  'text': 'Canon EOS R8 zdobył uznanie użytkowników dzięki swojej świetnej jakości zdjęć, ergonomii i nowoczesnym funkcjom. Użytkownicy doceniają jego wszechstronność, lekką konstrukcję oraz nowoczesny autofocus, co czyni go idealnym dla amatorów i profesjonalistów. Mimo drobnych sugestii dotyczących baterii, ogólne wrażenia są bardzo pozytywne.',
  'recommended': None,
  'date': None,
  'usage_period': None,
  'pros': ['Jakość zdjęć', 'Ergonomia', 'Funkcjonalność', 'Design'],
  'votes_yes': 1,
  'votes_no': 1},
 {'score': '5/5',
  'text': "Krótka ocena fotografa amatora po przesiadce z EOS'a RP:\nErgonomia prawie taka sama, poza włącznikiem, który teraz można obsługiwać jedną ręką - moim zdaniem zdecydowanie na plus.\nPo recenzjach spodziewałem się kosmicznych osiągów nowego AF i tu się trochę zawiodłem. Jest lepiej ale AF w RP nie był aż taki kulawy, a ten nie jest, aż taki genialny.\nZa to w kwestii, w której najbardziej zaskoczył mnie R8 a czego się nie spodziewałe

In [4]:
# Debug: inspect what Ceneo returns on later pages
from pathlib import Path

def debug_fetch_pages(product_id: str, pages=(1, 2)):
    session = load_or_create_session(product_id)
    session.get(f"{BASE_URL}/{product_id}", timeout=10)

    for p in pages:
        url = f"{BASE_URL}/{product_id}/opinie-{p}"
        r = session.get(url, timeout=20, allow_redirects=True)
        html = r.text

        title = None
        try:
            title = BeautifulSoup(html, "html.parser").title.get_text(strip=True)
        except Exception:
            pass

        out = Path(f"debug_opinie_{p}.html")
        out.write_text(html, encoding="utf-8", errors="ignore")

        print(f"\n--- opinie-{p} ---")
        print("status:", r.status_code)
        print("final_url:", r.url)
        print("bytes:", len(r.content))
        print("title:", title)
        print("js_product-review count:", html.count("js_product-review"))
        markers = ["captcha", "robot", "cloudflare", "access denied", "weryfik", "przeglądark", "zablok", "blocked"]
        low = html.lower()
        found = [m for m in markers if m in low]
        if found:
            print("markers:", found)


debug_fetch_pages("148187174", pages=(1,2,3,4))

Loaded existing cookies

--- opinie-1 ---
status: 200
final_url: https://www.ceneo.pl/148187174#tab=reviews
bytes: 607521
title: Aparat cyfrowy Canon EOS R8 bezlusterkowiec - Ceny i opinie na Ceneo.pl
js_product-review count: 110
markers: ['captcha', 'robot', 'przeglądark']

--- opinie-2 ---
status: 200
final_url: https://www.ceneo.pl/Captcha/Add?ReturnUrl=%252f148187174%252fopinie-2&CookieAge=172800
bytes: 16570
title: None
js_product-review count: 0
markers: ['captcha', 'robot', 'cloudflare']

--- opinie-3 ---
status: 200
final_url: https://www.ceneo.pl/Captcha/Add?ReturnUrl=%252f148187174%252fopinie-3&CookieAge=172800
bytes: 16570
title: None
js_product-review count: 0
markers: ['captcha', 'robot', 'cloudflare']

--- opinie-4 ---
status: 200
final_url: https://www.ceneo.pl/Captcha/Add?ReturnUrl=%252f148187174%252fopinie-4&CookieAge=172800
bytes: 16570
title: None
js_product-review count: 0
markers: ['captcha', 'robot', 'cloudflare']
